# FROST tabular observation graph

Build and plot a graph for **one FROST observation window**, following `training_configs/dop.yaml`: `AnemoiDatasetNodes`, a triangular hidden mesh, reversed KNN encoder edges, multiscale processor edges, and KNN decoder edges.

Select this repository's `.venv/bin/python` as the notebook kernel. Set `frost_path` below to the **tabular** Zarr store (or set `FROST_DATASET` before starting Jupyter). The shared `frost-rainrate-2024.zarr` uses the gridded layout and is not the tabular input for this notebook.

Each observation row becomes a node. Repeated observations at a station share coordinates and overlap in the plot. Only the selected window is read; the resulting graph is a snapshot for inspection.

Environment requirement: Earthkit needs Python’s `sqlite3` module when loading tabular data. If loading raises `ModuleNotFoundError: sqlite3`, the kernel needs a Python installation with SQLite support.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import zarr
from anemoi.datasets import open_dataset
from anemoi.graphs.create import GraphCreator
from anemoi.graphs.nodes import AnemoiDatasetNodes
from hydra.utils import instantiate
from omegaconf import OmegaConf
from torch_geometric.data import HeteroData

os.environ["ANEMOI_GRAPHS_FORCE_CPU"] = "1"
repo_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "training_configs/dop.yaml").is_file()
)
dop = OmegaConf.load(repo_root / "training_configs/dop.yaml")

In [ ]:
frost_path = os.environ.get("FROST_DATASET", "/path/to/frost-tabular.zarr")
frequency = dop.data.frequency
window = "(-3h,+0h]"
window_index = 1
hidden_resolution = dop.graph.nodes.hidden.node_builder.resolution
save_path = None  # Set to a new .pt path to save this window's graph.

## Read one observation window

`frequency` sets the spacing between reference times; `window` selects observations relative to each reference time. Change `window_index` to inspect another time. Index 1 avoids the usually partial window at the start of the store. Empty windows stop with a message so you can select another index.

In [ ]:
if not Path(frost_path).is_dir():
    raise FileNotFoundError("Set frost_path or FROST_DATASET to the FROST tabular Zarr store.")
store = zarr.open(frost_path, mode="r")
if store.attrs.get("layout") != "tabular":
    raise ValueError(f"Expected a tabular FROST store: {frost_path}")

dataset_config = OmegaConf.create({
    "dataset": frost_path,
    "frequency": frequency,
    "window": window,
})
dataset = open_dataset(OmegaConf.to_container(dataset_config, resolve=True))
observations = dataset[window_index]
if len(observations) == 0:
    raise ValueError("This window contains no observations; choose another window_index.")

reference_date = observations.reference_date
print(f"Reference time: {reference_date}; window: {window}")
print(f"Observation nodes: {len(observations):,}")
print(f"Variables: {dataset.variables}")

## Reuse the DOP graph settings

Use the microwave branch as the template, replace its dataset with FROST, and retain its encoder/decoder settings plus the hidden mesh and processor edges. The printed configuration shows the corresponding FROST graph declaration.

In [ ]:
template = OmegaConf.to_container(dop.graph, resolve=True)
frost_node = template["nodes"]["microwave"]
frost_node["node_builder"]["dataset"] = OmegaConf.to_container(dataset_config)
hidden_node = template["nodes"]["hidden"]
hidden_node["node_builder"]["resolution"] = hidden_resolution
edges = []
for edge in template["edges"]:
    if {edge["source_name"], edge["target_name"]} <= {"microwave", "hidden"}:
        for key in ("source_name", "target_name"):
            if edge[key] == "microwave":
                edge[key] = "frost"
        for builder in edge["edge_builders"]:
            if "scale_resolutions" in builder:
                builder["scale_resolutions"] = hidden_resolution
        edges.append(edge)

graph_config = OmegaConf.create({
    "nodes": {"frost": frost_node, "hidden": hidden_node},
    "edges": edges,
})
print(OmegaConf.to_yaml(graph_config))

## Build the graph for this window

**Compatibility detail:** in the installed Anemoi version, tabular datasets expose latitude and longitude on each observation window, while dataset-level coordinates are `None`. Training separates these dynamic nodes from the static graph. Calling `AnemoiDatasetNodes.update_graph()` directly on a tabular store therefore fails.

For this inspection graph, instantiate the configured `AnemoiDatasetNodes` and use its `reshape_coords()` on the selected window to obtain Anemoi's `(latitude, longitude)` coordinates in radians. `GraphCreator` then adds the hidden mesh and spatial edges. The `Timedeltas` attributes in `dop.yaml` are runtime attributes; compute them from this window's offsets in seconds.

In [ ]:
node_builder = instantiate(graph_config.nodes.frost.node_builder, name="frost")
assert isinstance(node_builder, AnemoiDatasetNodes)
graph = HeteroData()
graph["frost"].x = node_builder.reshape_coords(
    observations.latitudes, observations.longitudes,
).to(dtype=torch.float32, device="cpu")
graph["frost"].node_type = type(node_builder).__name__
if not torch.isfinite(graph["frost"].x).all():
    raise ValueError("The selected window contains invalid coordinates.")

timedeltas = torch.as_tensor(observations.timedeltas, dtype=torch.float32)
graph["frost"].timedeltas = timedeltas
graph["frost"].timedelta = instantiate(
    graph_config.nodes.frost.attributes.timedelta
).compute(timedeltas)

spatial_config = OmegaConf.create(OmegaConf.to_container(graph_config, resolve=True))
del spatial_config.nodes.frost
creator = GraphCreator(spatial_config)
graph = creator.update_graph(graph)
graph = creator.clean(graph)

graph.validate(raise_on_error=True)
print(graph)

## Plot the observation and hidden nodes

The left panel shows FROST nodes coloured by their time offset. The right panel shows the hidden mesh within the same extent. Longitude is wrapped to `[-180, 180)` for plotting; graph coordinates retain the Anemoi convention. These longitude/latitude plots work without downloading map data.

In [ ]:
frost_coords = np.rad2deg(graph["frost"].x.cpu().numpy())
hidden_coords = np.rad2deg(graph["hidden"].x.cpu().numpy())
frost_lon = (frost_coords[:, 1] + 180) % 360 - 180
hidden_lon = (hidden_coords[:, 1] + 180) % 360 - 180
lat_min, lat_max = frost_coords[:, 0].min(), frost_coords[:, 0].max()
lon_min, lon_max = frost_lon.min(), frost_lon.max()
padding = 2.0

fig, axes = plt.subplots(1, 2, figsize=(14, 7), constrained_layout=True)
points = axes[0].scatter(
    frost_lon, frost_coords[:, 0], c=timedeltas.numpy() / 3600,
    s=10, cmap="viridis", alpha=0.7, rasterized=True,
)
fig.colorbar(points, ax=axes[0], label="Time offset from reference (hours)")
axes[0].set_title(f"FROST: {graph['frost'].num_nodes:,} observation nodes")
axes[1].scatter(hidden_lon, hidden_coords[:, 0], s=10, color="tab:orange")
axes[1].scatter(frost_lon, frost_coords[:, 0], s=3, color="tab:blue", alpha=0.3)
axes[1].set_title(f"Hidden mesh (resolution {hidden_resolution}) with FROST nodes")
for ax in axes:
    ax.set_xlim(lon_min - padding, lon_max + padding)
    ax.set_ylim(max(-90, lat_min - padding), min(90, lat_max + padding))
    ax.set_xlabel("Longitude (degrees)")
    ax.set_ylabel("Latitude (degrees)")
    ax.grid(alpha=0.25)
fig.suptitle(f"FROST observation graph — {reference_date} — {window}")
plt.show()

In [ ]:
if save_path is not None:
    output = Path(save_path).expanduser()
    if output.exists():
        raise FileExistsError(f"Choose a new save_path: {output}")
    creator.save(graph, output)
    print(f"Saved window graph to {output}")